# Herramienta Interactiva de Visualización y Análisis Espaciotemporal 3D de la Sismicidad en Chile
### Cuaderno de Trabajo y Experimentación: Caso de Estudio Terremoto de Illapel 2015 ($M_w = 8,3$)

**Asignatura:** EIN092B Visualización de Datos  
**Profesor:** Jorge Portilla  
**Autores:** Fabian Acuña (`facunaa@usm.cl`), Fernando Gutiérrez (`fgutierrezp@usm.cl`)  

---

## Propósito y Estructura General del Proyecto
Este cuaderno documenta y ejecuta todo el flujo de ciencia de datos, entrenamiento y evaluación del modelo de agrupamiento no supervisado (**DBSCAN 3D**), el análisis de sensibilidad frente a **K-Means**, el ajuste temporal de la **ley de Omori-Utsu** y la generación de las figuras oficiales para el informe en formato IEEE.

---
## Etapa 1: Adquisición Automática y Reproducible de Datos (API USGS)
- Consulta a la API pública del USGS entre el **15 de septiembre de 2015** y el **15 de octubre de 2015**.
- Región centro-norte de Chile: Latitud `[-35.0, -28.0]`, Longitud `[-74.0, -69.0]`, Magnitud mínima $M \ge 3,0$.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.optimize import curve_fit

USGS_API_URL = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    "&starttime=2015-09-15"
    "&endtime=2015-10-15"
    "&minmagnitude=3.0"
    "&minlatitude=-35.0"
    "&maxlatitude=-28.0"
    "&minlongitude=-74.0"
    "&maxlongitude=-69.0"
)
CSV_FILENAME = "illapel_2015_usgs.csv"

if not os.path.exists(CSV_FILENAME):
    print("Descargando catálogo directamente desde la API oficial de USGS...")
    df_raw = pd.read_csv(USGS_API_URL)
    df_raw.to_csv(CSV_FILENAME, index=False)
    print(f"Descarga completada: '{CSV_FILENAME}'")
else:
    print(f"Cargando catálogo local versionado: '{CSV_FILENAME}'")
    df_raw = pd.read_csv(CSV_FILENAME)

print(f"Total de registros brutos: {len(df_raw)}")
df_raw.head(3)

---
## Etapa 2: Exploración, Limpieza y Estructuración del Conjunto de Datos
- Selección de variables (`datetime`, `latitude`, `longitude`, `depth`, `mag`, `place`), estandarización temporal a UTC, eliminación de nulos y duplicados, e identificación del evento principal (*Mainshock* $M_w = 8,3$).

In [2]:
df = df_raw.copy()
df['datetime'] = pd.to_datetime(df['time'])

cols_interes = ['datetime', 'latitude', 'longitude', 'depth', 'mag', 'place']
df = df[cols_interes].dropna().drop_duplicates().sort_values('datetime').reset_index(drop=True)

idx_principal = df['mag'].idxmax()
mainshock = df.loc[idx_principal]

print("=== SISMO PRINCIPAL DETECTADO (ILLAPEL 2015) ===")
print(f"Fecha y Hora (UTC): {mainshock['datetime']}")
print(f"Magnitud: Mw {mainshock['mag']}")
print(f"Epicentro: Latitud {mainshock['latitude']}°, Longitud {mainshock['longitude']}°")
print(f"Profundidad Hipocentral: {mainshock['depth']} km")
print(f"Referencia geográfica: {mainshock['place']}")
print("-" * 50)
print(f"Total de eventos limpios y válidos (N): {len(df)}")
df[['latitude', 'longitude', 'depth', 'mag']].describe().round(2)

---
## Etapa 3: Normalización Espacial 3D y Selección de Hiperparámetros ($k$-Distance Graph)
- Estandarización mediante puntaje $Z$ (`StandardScaler`) de las coordenadas `[latitude, longitude, depth]`.
- Cálculo de la curva de distancias al octavo vecino más cercano ($k = MinPts = 8$, según la heurística $MinPts \ge 2D$ de Sander et al., 1998) para determinar objetivamente el radio $\varepsilon = 0,45$ en el punto de inflexión (método del codo).

In [3]:
X_coords = df[['latitude', 'longitude', 'depth']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_coords)

k = 8
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, k - 1])

plt.figure(figsize=(6.5, 3.8), dpi=300)
plt.plot(k_distances, color="#1f77b4", lw=2, label=f"Distancia al k-vecino (k={k})")
plt.axhline(y=0.45, color="red", linestyle="--", label=r"Umbral seleccionado ($\varepsilon = 0,45$)")
plt.xlabel("Índice de eventos ordenados", fontsize=10)
plt.ylabel(f"Distancia al {k}-ésimo vecino", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig("figura2_k_distance_graph.png", dpi=300)
plt.show()

---
## Etapa 4: Entrenamiento de DBSCAN, Análisis de Sensibilidad y Comparación con K-Means
- Evaluación sistemática del modelo variando $\varepsilon \in \{0,35; 0,45; 0,55\}$ y $MinPts \in \{6, 8, 10\}$.
- Comparación empírica frente a **K-Means** ($k = 2$ y $k = 3$) cuantificando número de grupos, porcentaje agrupado, filtrado de ruido, profundidad máxima del grupo principal y coeficiente de silueta ($S$).

In [4]:
# 1. Tabla comparativa de sensibilidad y algoritmos
resultados_modelos = []
configuraciones_dbscan = [(0.35, 6), (0.35, 8), (0.45, 8), (0.45, 10), (0.55, 8)]

for eps_val, mp_val in configuraciones_dbscan:
    labels = DBSCAN(eps=eps_val, min_samples=mp_val).fit_predict(X_scaled)
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    n_agrup = len(df) - n_noise
    sil = silhouette_score(X_scaled, labels)
    prof_max_c0 = df[labels == 0]['depth'].max() if 0 in labels else np.nan
    resultados_modelos.append({
        'Modelo / Configuración': f'DBSCAN (eps={eps_val}, MinPts={mp_val})',
        'Grupos': n_cl,
        'Agrupados (%)': f'{n_agrup} ({n_agrup/len(df)*100:.1f}%)',
        'Ruido (%)': f'{n_noise} ({n_noise/len(df)*100:.1f}%)',
        'Prof. Máx. C0 (km)': round(prof_max_c0, 1),
        'Silueta (S)': round(sil, 3)
    })

for k_val in [2, 3]:
    km_labels = KMeans(n_clusters=k_val, random_state=42, n_init=10).fit_predict(X_scaled)
    sil_km = silhouette_score(X_scaled, km_labels)
    resultados_modelos.append({
        'Modelo / Configuración': f'K-Means (k={k_val})',
        'Grupos': k_val,
        'Agrupados (%)': f'{len(df)} (100.0%)',
        'Ruido (%)': '0 (0.0%)',
        'Prof. Máx. C0 (km)': round(df['depth'].max(), 1),
        'Silueta (S)': round(sil_km, 3)
    })

df_sensibilidad = pd.DataFrame(resultados_modelos)
print("=== TABLA I: ANÁLISIS DE SENSIBILIDAD Y COMPARACIÓN CON K-MEANS ===")
display(df_sensibilidad)

# 2. Asignación final con la configuración óptima (eps=0.45, MinPts=8)
dbscan_optimo = DBSCAN(eps=0.45, min_samples=8)
df['cluster'] = dbscan_optimo.fit_predict(X_scaled)

---
## Etapa 5: Modelado Temporal de la Secuencia de Réplicas (Ley de Omori-Utsu)
- Agrupación de las $718$ réplicas posteriores al sismo principal en intervalos diarios ($\Delta t = 1\text{ día}$) y ajuste por mínimos cuadrados no lineales (`scipy.optimize.curve_fit`) de la ley modificada de Omori-Utsu: $n(t) = K / (t + c)^p$.

In [5]:
t0 = mainshock['datetime']
df_replicas = df[df['datetime'] >= t0].copy()
df_replicas['dias_desde_mainshock'] = (df_replicas['datetime'] - t0).dt.total_seconds() / 86400.0
df_replicas['dia_entero'] = np.floor(df_replicas['dias_desde_mainshock']).astype(int)

conteo_diario = df_replicas.groupby('dia_entero').size()
t_vals = conteo_diario.index.values + 0.5
n_vals = conteo_diario.values

def ley_omori_utsu(t, K, c, p):
    return K / ((t + c) ** p)

popt, _ = curve_fit(ley_omori_utsu, t_vals, n_vals, p0=[150, 0.5, 1.0], bounds=([1, 0.01, 0.3], [1000, 5.0, 2.5]))
K_est, c_est, p_est = popt
n_pred = ley_omori_utsu(t_vals, *popt)
r2_omori = 1 - np.sum((n_vals - n_pred)**2) / np.sum((n_vals - np.mean(n_vals))**2)

print("=== RESULTADOS DEL AJUSTE TEMPORAL DE OMORI-UTSU ===")
print(f"Réplicas en primeras 24 h: {conteo_diario.get(0, 0)} ({conteo_diario.get(0, 0)/len(df_replicas)*100:.1f}%)")
print(f"Réplicas en primera semana (7 días): {conteo_diario.loc[0:6].sum()} ({conteo_diario.loc[0:6].sum()/len(df_replicas)*100:.1f}%)")
print(f"Parámetros estimados: K = {K_est:.1f}, c = {c_est:.2f} días, p = {p_est:.2f} (R² = {r2_omori:.3f})")

---
## Etapa 6: Generación de las Figuras Oficiales en Alta Resolución (300 DPI)

In [6]:
mag_main = mainshock['mag']

# FIGURA 1: Mapa 2D de epicentros
plt.figure(figsize=(5.8, 5.8), dpi=300)
sc = plt.scatter(df['longitude'], df['latitude'], c=df['depth'], cmap='viridis_r', s=df['mag']**2.2 * 1.4, alpha=0.65, edgecolors='k', linewidth=0.3)
plt.scatter(mainshock['longitude'], mainshock['latitude'], color='red', marker='*', s=320, edgecolors='black', linewidth=1.1, label=f'Sismo Principal (Mw {mag_main})')
cbar = plt.colorbar(sc)
cbar.set_label('Profundidad Hipocentral (km)', fontsize=10)
plt.xlabel('Longitud (°O)', fontsize=10)
plt.ylabel('Latitud (°S)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc='lower left', frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig('figura1_mapa_epicentros_2d.png', dpi=300)
plt.show()

# FIGURA 3: Perfil 3D de hipocentros
fig = plt.figure(figsize=(7.2, 5.2), dpi=300)
ax = fig.add_subplot(111, projection='3d')
p = ax.scatter(df['longitude'], df['latitude'], -df['depth'], c=df['depth'], cmap='inferno', s=df['mag']**2 * 1.1, alpha=0.65)
ax.scatter([mainshock['longitude']], [mainshock['latitude']], [-mainshock['depth']], color='cyan', marker='*', s=360, edgecolors='black', linewidth=1.1, label=f'Hipocentro Mw {mag_main}')
ax.set_xlabel('Longitud (°O)', fontsize=9)
ax.set_ylabel('Latitud (°S)', fontsize=9)
ax.set_zlabel('Profundidad (-km)', fontsize=9)
fig.colorbar(p, ax=ax, label='Profundidad (km)', shrink=0.6, pad=0.08)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('figura3_perfil_subduccion_3d.png', dpi=300)
plt.show()

# FIGURA 4: Segmentación DBSCAN
plt.figure(figsize=(5.8, 5.8), dpi=300)
df_noise = df[df['cluster'] == -1]
df_clusters = df[df['cluster'] != -1]
plt.scatter(df_noise['longitude'], df_noise['latitude'], c='gray', s=18, alpha=0.4, label='Ruido / Sismicidad dispersa (99)')
plt.scatter(df_clusters['longitude'], df_clusters['latitude'], c='#1f77b4', s=df_clusters['mag']**2.2 * 1.4, alpha=0.8, edgecolors='black', linewidth=0.35, label='Cluster 0: Zona de ruptura (620)')
plt.scatter(mainshock['longitude'], mainshock['latitude'], color='red', marker='*', s=320, edgecolors='black', linewidth=1.1, label=f'Sismo Principal (Mw {mag_main})')
plt.xlabel('Longitud (°O)', fontsize=10)
plt.ylabel('Latitud (°S)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc='lower left', frameon=True, fontsize=8.5)
plt.tight_layout()
plt.savefig('figura4_clusters_dbscan.png', dpi=300)
plt.show()

---
## Etapa 7: Documentación y Arquitectura del Dashboard Interactivo (`app.py`)
Una vez validados los modelos en este cuaderno, el flujo de preprocesamiento, el agrupamiento DBSCAN ($\varepsilon = 0,45$, $MinPts = 8$) y las vistas coordinadas (Mapa 2D en OpenStreetMap, Visor 3D de hipocentros y evolución temporal) se integran en la aplicación interactiva `app.py`, ejecutable mediante:
```bash
python -m streamlit run app.py
```